# v33 — Verifier-Gated Reading Arbitration, on Google Colab

The same confirmation as `MAS_SHT_Kaggle_v33.ipynb`; rule in `gated_reading.py`, pre-registration in `V33_PLAN.md`.

**The result file lives on Google Drive** (`MyDrive/MAS_SHT_v33/pretest_v25.json`) and is saved after every row. If the session ends, open the notebook again and **Run all**: it continues where it stopped.

**Before running**
1. Runtime → Change runtime type → **T4 GPU**.
2. Secrets (key icon) with *Notebook access* on: `HF_API_KEY`.
3. Runtime → **Run all**. Keep the tab open.

**How long:** with one GPU every row is drawn first and scored afterwards; expect 12+ h, i.e. several Colab sessions. Kaggle (two T4s) is the better fit.


In [ ]:
# === Cell 1 — Colab setup ===========================================================
import os, subprocess, sys

from google.colab import drive
drive.mount('/content/drive')
OUT_DIR = '/content/drive/MyDrive/MAS_SHT_v33'
os.makedirs(OUT_DIR, exist_ok=True)

# The stack of the Kaggle runs. Colab's own torch is kept.
DEPS = ['transformers>=4.44.0,<4.50', 'accelerate', 'bitsandbytes', 'openai>=1.40.0',
        'python-dotenv', 'tabulate']
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + DEPS)
print('Deps installed.')

def _secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)

_hf = _secret('HF_API_KEY')
if _hf:
    for _k in ('HF_API_KEY', 'HUGGINGFACE_TOKEN', 'HF_TOKEN'):
        os.environ[_k] = _hf
    print('HF token loaded.')

import torch
if not torch.cuda.is_available():
    raise RuntimeError('No GPU: Runtime -> Change runtime type -> T4 GPU, then run again.')
_p = torch.cuda.get_device_properties(0)
print(f'cuda:0 {_p.name}  sm_{_p.major}{_p.minor}  {_p.total_memory / 2**30:.0f} GB')
if (_p.major, _p.minor) < (7, 5):
    raise RuntimeError(f'{_p.name} is older than sm_75; 4-bit NF4 needs a T4 or newer.')

REPO_DIR = '/content/llm_thesis'
BRANCH   = 'main'
_tok = _secret('GITHUB_TOKEN')
_url = f'https://{_tok}@github.com/marios4371/llm_thesis.git' if _tok else 'https://github.com/marios4371/llm_thesis.git'
if not os.path.isdir(REPO_DIR):
    _r = subprocess.run(['git', 'clone', _url, REPO_DIR], capture_output=True, text=True)
    if _r.returncode:
        raise RuntimeError('git clone failed:\n' + _r.stderr.replace(_tok or '\0', '***'))
subprocess.run(['git', '-C', REPO_DIR, 'remote', 'set-url', 'origin', _url], capture_output=True)
for _args in (['fetch', 'origin', BRANCH], ['checkout', '-B', BRANCH, f'origin/{BRANCH}']):
    _r = subprocess.run(['git', '-C', REPO_DIR, *_args], capture_output=True, text=True)
    if _r.returncode:
        raise RuntimeError(f"git {' '.join(_args)} failed:\n" + _r.stderr.replace(_tok or '\0', '***'))
os.chdir(REPO_DIR)
print('code at', subprocess.run(['git', 'log', '-1', '--format=%h %s'], capture_output=True, text=True).stdout.strip())
OUT = f'{OUT_DIR}/pretest_v25.json'
print('results go to', OUT)


In [ ]:
# === Cell 2 — offline tests (a few minutes) ============================================
import subprocess, sys
for _test in ('test_v25.py', 'test_v33.py'):
    _t = subprocess.run([sys.executable, _test], capture_output=True, text=True)
    print(_test, '->', _t.stdout.strip().splitlines()[-1] if _t.stdout.strip() else _t.stderr[-2000:])
    if _t.returncode != 0:
        print(_t.stdout[-5000:], _t.stderr[-3000:])
        raise RuntimeError(f'{_test} failed: do NOT start the GPU run')


In [ ]:
# === Cell 3 — the confirmation (resumes from the file on Drive) ========================
import subprocess, sys
CMD = [sys.executable, '-u', 'pretest_v33.py', '--out', OUT, '--max-hours', '11.0']
_p = subprocess.Popen(CMD, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for _line in _p.stdout:
    print(_line, end='')
_p.wait()
print('exit code', _p.returncode)


In [ ]:
# === Cell 4 — the pre-registered reading (no GPU) ======================================
!python -u pretest_v33.py --summary-only --out "$OUT"
